# 第6章 收益率与财富路径

> **核心问题**：价格变化怎样转化为可比较的回报？为什么平均收益不错，最终财富仍可能令人失望？

- 金融线：价格收益、总回报、累计收益、年化、定投和路径依赖。
- 数学线：比率、连乘、对数求和、算术平均与几何平均。
- Python线：`shift`、`pct_change`、`cumprod`、向量化、时间索引和函数测试。

## AI学习状态

当前进度：第6章开始  
已掌握：价格、股息、时间索引和数据审计  
仍然薄弱：待填写  
下一步：每个收益率都写清起止时点和现金流。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"]=(8,4.5); plt.rcParams["axes.grid"]=True
plt.rcParams["font.sans-serif"]=["Arial Unicode MS","PingFang SC","SimHei","DejaVu Sans"]
plt.rcParams["axes.unicode_minus"]=False
rng=np.random.default_rng(20260711)

## 6.1 简单收益率

$$R_t=\frac{P_t-P_{t-1}}{P_{t-1}}=\frac{P_t}{P_{t-1}}-1$$

它是相对于期初价格的比例。上涨10%后再下跌10%不会回到原点，因为第二个10%的基数不同。

In [ ]:
prices=pd.Series([100,110,99],index=pd.date_range("2026-01-01",periods=3,freq="D"),name="price")
returns=prices.pct_change()
display(pd.concat([prices,returns.rename("return")],axis=1).style.format({"price":"{:.2f}","return":"{:.2%}"}))
print("最终累计收益：",f"{prices.iloc[-1]/prices.iloc[0]-1:.2%}")

**Python提示：`pct_change()`**计算当前值相对前一期的比例变化，第一行因没有前一期而是`NaN`。它不会判断价格是否已复权，也不会自动加入股息。

### 我的解释

为什么+10%和-10%的算术和为0，但财富变化为-1%？

<!-- 在这里填写；完成前AI不要代答 -->

## 6.2 总回报与累计财富

有股息 $D_t$ 时，一期总回报为

$$R_t^{total}=\frac{P_t-P_{t-1}+D_t}{P_{t-1}}$$

多期财富通过连乘累积：

$$W_T=W_0\prod_{t=1}^T(1+R_t)$$

In [ ]:
data=pd.DataFrame({"price":[100,103,101,106],"dividend":[0,0,2,0]},
                  index=pd.date_range("2026-03-01",periods=4,freq="ME"))
data["price_return"]=data["price"].pct_change()
data["total_return"]=(data["price"]+data["dividend"])/data["price"].shift(1)-1
data["wealth_price_only"]=10_000*(1+data["price_return"].fillna(0)).cumprod()
data["wealth_total"]=10_000*(1+data["total_return"].fillna(0)).cumprod()
data

**量化编程警告**：股息的除息时点、再投资价格和税费会影响真实总回报。把股息简单加到收盘价只适合本章的期末教学例子。

## 6.3 对数收益率

$$r_t=\ln\left(\frac{P_t}{P_{t-1}}\right)=\ln(1+R_t)$$

对数收益可跨期相加，但把相加结果转回简单累计收益时要用 $e^{\sum r_t}-1$。当简单收益接近-100%时，对数收益会非常负；价格不能为非正数。

In [ ]:
simple=prices.pct_change().dropna()
log_return=np.log(prices/prices.shift(1)).dropna()
print({"简单收益连乘":float((1+simple).prod()-1),
       "对数收益求和后转换":float(np.exp(log_return.sum())-1),
       "对数收益之和":float(log_return.sum())})

## 6.4 算术平均不等于长期增长率

算术平均描述单期收益的平均；几何平均描述从初值到终值的等效复合增长率。波动越大，两者差距通常越明显。

In [ ]:
samples={"稳定":[0.05,0.05],"波动":[0.50,-0.40],"先跌后涨":[-0.40,0.50]}
rows=[]
for name,rs in samples.items():
    rs=np.array(rs,dtype=float)
    rows.append({"路径":name,"算术平均":rs.mean(),"几何平均":(np.prod(1+rs))**(1/len(rs))-1,"终值":100*np.prod(1+rs)})
display(pd.DataFrame(rows).set_index("路径").style.format({"算术平均":"{:.2%}","几何平均":"{:.2%}","终值":"{:.2f}"}))

In [ ]:
vols=np.linspace(0,0.50,100)
mean=0.08
approx_growth=mean-0.5*vols**2
plt.plot(vols,approx_growth); plt.axhline(0,color="black",lw=1)
plt.xlabel("波动率"); plt.ylabel("近似对数增长率"); plt.title("固定算术均值下的波动拖累（近似）"); plt.show()

### 观察问题

为什么“先跌后涨”和“先涨后跌”的无追加终值相同？如果中间有定投或提款，顺序还会无关吗？

### 我的回答

<!-- 在这里填写；完成前AI不要代答 -->

## 6.5 年化必须说明频率和样本长度

日均收益简单乘252是一种近似；复合年化常写为`(终值/初值) ** (每年期数/样本期数) - 1`。252只是常用交易日近似，不适合所有市场、资产或缺失数据。

In [ ]:
monthly_returns=np.array([.02,-.01,.03,.00,.015,-.02,.01,.025,-.005,.02,.01,.015])
annual_compound=np.prod(1+monthly_returns)-1
annual_arithmetic=monthly_returns.mean()*12
print({"复合年度收益":f"{annual_compound:.2%}","月均收益×12":f"{annual_arithmetic:.2%}"})

## 6.6 定投引入现金流，不能只看资产收益率

资金加权收益会受现金流时点影响；时间加权收益用于隔离外部现金流影响。本节先模拟财富，不急于引入完整绩效归因。

In [ ]:
def wealth_with_contributions(returns,initial=0,contribution=1000):
    wealth=initial; path=[wealth]
    for r in returns:
        wealth=wealth*(1+r)+contribution  # 期末追加
        path.append(wealth)
    return np.array(path)

path_a=wealth_with_contributions([-.30,.40,.10])
path_b=wealth_with_contributions([.10,.40,-.30])
pd.DataFrame({"先跌路径":path_a,"后跌路径":path_b},index=range(4))

### 我的解释

两组收益包含相同三个数字，为什么定投终值不同？“下跌对长期投资者一定有利”这句话遗漏了哪些风险？

<!-- 在这里填写；完成前AI不要代答 -->

## 6.7 编程练习：累计财富

函数接收收益率和初始财富；拒绝任何`return <= -1`；返回包含初始值的数组。

In [ ]:
def cumulative_wealth(returns,initial=1.0):
    # TODO
    return None

In [ ]:
ans=cumulative_wealth([.10,-.10],100)
if ans is None: print("练习尚未完成。")
else: print("测试通过：",np.allclose(ans,[100,110,99]))

## 本章总结与小项目

为一组含价格和股息的月度数据计算价格收益、总回报、累计财富、算术/几何平均和复合年化；再加入三种现金流时点，解释路径差异。

**底线**：收益率定义、现金流、复权、频率和年化口径必须同时报告。